# CSE 151B Competition — Optimized Notebook

Key improvements over the starter:
- LaTeX repair (restore `\frac`, `\infty`, `\int`, etc. before prompting)
- MCQ logprobs calibration pass (reads P(letter) directly, no text parsing needed)
- Better MCQ/free-form system prompts tuned for the thinking model
- Stronger `extract_letter` (abstains instead of guessing a random capital)
- Free-form `\boxed{}` extraction from last match in response

## 1. Environment Setup

Same as starter. Comment out the install block after first run, then restart the kernel.

In [ ]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create venv and install — use full path since PATH isn't updated yet
# !~/.local/bin/uv venv .venv --seed
# !~/.local/bin/uv pip install --python .venv/bin/python sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")

In [ ]:
# Activate venv — run every time
!source ./.venv/bin/activate

## 2. Imports & Configuration

In [ ]:
import json
import os
import re
import sys
from collections import Counter
from pathlib import Path
from typing import Optional

import sympy as sp
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

MODEL_ID     = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID       = "0"
DATA_PATH    = "data/public.jsonl"
OUTPUT_PATH  = "results/optimized_results.jsonl"
MAX_TOKENS   = 4096
N_QUESTIONS  = 10     # set to len(data) for the full dataset

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 3. Load Dataset

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")
print(f"Will evaluate first {N_QUESTIONS} questions.")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

## 4. LaTeX Repair

The dataset stores `frac{a}{b}`, `infty`, `int`, `sin`, etc. without backslashes.
`repair_latex()` restores them before any text is sent to the model.

In [ ]:
# Fix #1: restore missing backslashes before LaTeX commands.
# Pattern: word-boundary match on the bare command, negative lookbehind to skip
# already-correct \frac etc.  "times" and "pi" are safe in all-math questions.
_LATEX_SUBS = [
    (r'(?<!\\)\bfrac\b',    r'\\frac'),
    (r'(?<!\\)\binfty\b',   r'\\infty'),
    (r'(?<!\\)\bint\b',     r'\\int'),
    (r'(?<!\\)\bsum\b',     r'\\sum'),
    (r'(?<!\\)\bprod\b',    r'\\prod'),
    (r'(?<!\\)\blim\b',     r'\\lim'),
    (r'(?<!\\)\bsqrt\b',    r'\\sqrt'),
    (r'(?<!\\)\bsin\b',     r'\\sin'),
    (r'(?<!\\)\bcos\b',     r'\\cos'),
    (r'(?<!\\)\btan\b',     r'\\tan'),
    (r'(?<!\\)\bcot\b',     r'\\cot'),
    (r'(?<!\\)\bsec\b',     r'\\sec'),
    (r'(?<!\\)\bcsc\b',     r'\\csc'),
    (r'(?<!\\)\blog\b',     r'\\log'),
    (r'(?<!\\)\bln\b',      r'\\ln'),
    (r'(?<!\\)\bexp\b',     r'\\exp'),
    (r'(?<!\\)\bpi\b',      r'\\pi'),
    (r'(?<!\\)\btheta\b',   r'\\theta'),
    (r'(?<!\\)\balpha\b',   r'\\alpha'),
    (r'(?<!\\)\bbeta\b',    r'\\beta'),
    (r'(?<!\\)\bgamma\b',   r'\\gamma'),
    (r'(?<!\\)\bdelta\b',   r'\\delta'),
    (r'(?<!\\)\bsigma\b',   r'\\sigma'),
    (r'(?<!\\)\blambda\b',  r'\\lambda'),
    (r'(?<!\\)\bmu\b',      r'\\mu'),
    (r'(?<!\\)\bepsilon\b', r'\\epsilon'),
    (r'(?<!\\)\bphi\b',     r'\\phi'),
    (r'(?<!\\)\bomega\b',   r'\\omega'),
    (r'(?<!\\)\bcdot\b',    r'\\cdot'),
    (r'(?<!\\)\bpm\b',      r'\\pm'),
    (r'(?<!\\)\bleq\b',     r'\\leq'),
    (r'(?<!\\)\bgeq\b',     r'\\geq'),
    (r'(?<!\\)\bneq\b',     r'\\neq'),
    (r'(?<!\\)\btimes\b',   r'\\times'),
    (r'(?<!\\)\bpartial\b', r'\\partial'),
    (r'(?<!\\)\bnabla\b',   r'\\nabla'),
]

_COMPILED_SUBS = [(re.compile(p), r) for p, r in _LATEX_SUBS]

def repair_latex(text: str) -> str:
    """Restore missing backslashes before LaTeX commands."""
    for pattern, replacement in _COMPILED_SUBS:
        text = pattern.sub(replacement, text)
    return text

# Smoke-test on the MCQ sample from the dataset
_test = "int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds"
print("Before:", _test)
print("After: ", repair_latex(_test))

## 5. Prompt Construction

In [ ]:
# Fix #8: MCQ prompt tuned for thinking model.
# The chain-of-thought lives in <think>...</think>. We don't suppress it —
# we just be unambiguous about what must follow </think>.
SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Think carefully through each option. "
    "After </think>, output exactly one line: \\boxed{X} "
    "where X is the single correct letter from the options (A, B, C, …). "
    "Nothing else after </think>."
)

# Fix #9: free-form prompt with two few-shot examples (2-answer and 3-answer)
# and an explicit single-\boxed{} rule.
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Use exactly one \\boxed{} containing all final answers separated by commas in order. "
    "Do NOT use multiple \\boxed{} blocks.\n\n"
    "Example — two answers:\n"
    "Problem: Find the roots of x^2 - 5x + 6 = 0.\n"
    "Solution: (x-2)(x-3)=0, so x=2 or x=3.\n"
    "\\boxed{2, 3}\n\n"
    "Example — three answers:\n"
    "Problem: A right triangle has legs a=3, b=4. Find hypotenuse c, perimeter P, and area A.\n"
    "Solution: c=5, P=3+4+5=12, A=½·3·4=6.\n"
    "\\boxed{5, 12, 6}"
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt); applies LaTeX repair to question text."""
    question = repair_latex(question)           # Fix #1: repair before sending
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(
            f"{lbl}. {repair_latex(opt.strip())}" for lbl, opt in zip(labels, options)
        )
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} system prompt ──")
    print(sys_p)
    print(f"\n── {label} user prompt (first 300 chars) ──")
    print(usr_p[:300], "...\n")

## 6. Load Model

Two generation passes for MCQ:
1. **Main pass** — full thinking chain, extract letter from text
2. **Logprobs pass** — cheap no-think pass, reads P(letter) directly over A–J tokens

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=True,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    enforce_eager=True,
    max_num_seqs=64,
)

_BASE = dict(temperature=0.7, top_p=0.8, top_k=20, min_p=0.0,
             presence_penalty=0.0, repetition_penalty=1.0)

main_sampling = SamplingParams(
    n=1, max_tokens=MAX_TOKENS,
    stop=["</answer>", "\n\nProblem", "\n\nQuestion"],
    **_BASE,
)

# Logprobs pass: near-greedy, no thinking, reads first-token distribution
logprob_sampling = SamplingParams(
    n=1, max_tokens=10,
    temperature=0.01, top_p=1.0,
    logprobs=20,
)

print("Model loaded.")

## 7. Generate Responses

In [ ]:
subset = data[:N_QUESTIONS]

def make_prompt(item: dict) -> str:
    system, user = build_prompt(item["question"], item.get("options"))
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False, add_generation_prompt=True,
    )

def make_calib_prompt(item: dict) -> str:
    """No-think prompt for MCQ logprobs pass."""
    _, user = build_prompt(item["question"], item.get("options"))
    try:
        return tokenizer.apply_chat_template(
            [{"role": "system", "content": "Output only the answer letter (A–J), nothing else."},
             {"role": "user",   "content": user}],
            tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        # Fallback if enable_thinking is not supported by this tokenizer version
        return tokenizer.apply_chat_template(
            [{"role": "system", "content": "Output only the answer letter (A–J), nothing else."},
             {"role": "user",   "content": user}],
            tokenize=False, add_generation_prompt=True,
        )

prompts = [make_prompt(item) for item in subset]

print(f"Running main generation for {len(prompts)} questions...")
outputs = llm.generate(prompts, main_sampling)
responses = [out.outputs[0].text.strip() for out in outputs]
print("Main generation done.")

# Logprobs calibration pass for MCQ only
mcq_mask = [bool(item.get("options")) for item in subset]
mcq_local = [i for i, is_mcq in enumerate(mcq_mask) if is_mcq]
calib_prompts = [make_calib_prompt(subset[i]) for i in mcq_local]

print(f"Running logprobs calibration for {len(calib_prompts)} MCQ questions...")
calib_outputs = llm.generate(calib_prompts, logprob_sampling) if calib_prompts else []
print("Calibration done.")

# Preview first 3 responses
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={subset[i].get('id')}) ──")
    print(responses[i][:300], "..." if len(responses[i]) > 300 else "")

## 8. Score Responses

In [ ]:
# Letter token IDs for logprobs scoring
letter_token_ids = {}
for _ch in "ABCDEFGHIJ":
    _ids = tokenizer.encode(_ch, add_special_tokens=False)
    if _ids:
        letter_token_ids[_ch] = _ids[0]

# High-confidence answer patterns checked before any fallback
_ANSWER_PATTERNS = [
    r"\\boxed\{([A-Ja-j])\}",
    r"(?:the\s+)?answer\s+is\s+[\(\[]?([A-J])[\)\]]?",
    r"(?:correct\s+)?answer\s*[:\-]\s*[\(\[]?([A-J])[\)\]]",
    r"(?:therefore|thus|so),?\s+(?:the\s+)?(?:answer|choice|option)\s+is\s+[\(\[]?([A-J])[\)\]]",
    r"option\s+([A-J])\s+is\s+correct",
]
_OPTION_MARKER = re.compile(r"^([A-J])[.)]\s", re.MULTILINE)

def extract_letter(text: str) -> str:
    for pat in _ANSWER_PATTERNS:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(1).upper()
    markers = _OPTION_MARKER.findall(text)
    return markers[-1].upper() if markers else ""  # abstain if nothing found

def score_mcq_logprobs(calib_out, options_count: int) -> str:
    """Return best letter from logprobs calibration output."""
    gen_text = calib_out.outputs[0].text.strip()
    valid = [chr(65 + i) for i in range(options_count)]
    if gen_text and gen_text[0].upper() in valid:
        return gen_text[0].upper()
    lps = calib_out.outputs[0].logprobs
    if lps:
        first_pos = lps[0]
        best_letter, best_lp = "", float("-inf")
        for letter in valid:
            tid = letter_token_ids.get(letter)
            if tid and tid in first_pos:
                lp = first_pos[tid].logprob
                if lp > best_lp:
                    best_lp, best_letter = lp, letter
        if best_letter:
            return best_letter
    return extract_letter(gen_text)

def extract_boxed(text: str) -> str:
    """Extract content from the last \\boxed{} in text."""
    matches = re.findall(r'\\boxed\{([^}]*)\}', text)
    return matches[-1].strip() if matches else ""

# Load judger
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

# Map MCQ calibration outputs back to subset indices
calib_map = {mcq_local[k]: calib_outputs[k] for k in range(len(calib_outputs))}

results = []
for idx, (item, response) in tqdm(enumerate(zip(subset, responses)),
                                   total=len(subset), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        calib_out = calib_map.get(idx)
        if calib_out is not None:
            pred = score_mcq_logprobs(calib_out, len(item["options"]))
        else:
            pred = extract_letter(response)
        correct = (pred == str(gold).strip().upper())
        meta = {"pred": pred}
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        answer = extract_boxed(response)
        try:
            correct = judger.auto_judge(pred=response, gold=gold_list,
                                        options=[[]] * len(gold_list))
        except Exception:
            correct = False
        meta = {"extracted": answer}

    results.append({"id": item.get("id"), "is_mcq": is_mcq,
                    "gold": gold, "response": response,
                    "correct": correct, **meta})

print(f"Scoring complete. {len(results)} results.")

## 9. Summary

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print(f"EVALUATION RESULTS (first {N_QUESTIONS} questions)")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 10. Save Results

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

- To run the full dataset: set `N_QUESTIONS = len(data)` in the config cell
- To re-enable majority voting: wrap `llm.generate` with `n=5` and aggregate with `Counter`
- If `enable_thinking=False` raises an error, the `try/except` in `make_calib_prompt` already falls back to a standard prompt automatically